In [109]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
from sklearn.metrics import r2_score, mean_absolute_error
import time

In [110]:
# data load
# Load driver statistics
driver_stats = pd.read_csv('./data/driver_features.csv')
total_drivers = len(driver_stats)

# Load race results and supplementary data
historical_races = pd.read_csv('data/f1_multi_season_results.csv')
constructor_data = pd.read_csv('data/teams_info.csv')
driver_stats = pd.read_csv('data/driver_features.csv')

In [111]:
# Circuit classification mapping
CIRCUIT_MAP = {
    'Australia': 'balanced', 'China': 'high_speed', 'Japan': 'technical',
    'Bahrain': 'high_speed', 'Saudi Arabia': 'high_speed', 'Miami': 'high_speed',
    'Emilia Romagna': 'technical', 'Monaco': 'technical', 'Spain': 'balanced',
    'Canada': 'balanced', 'Austria': 'technical', 'Great Britain': 'balanced',
    'Belgium': 'high_speed', 'Hungary': 'technical', 'Netherlands': 'technical',
    'Italy': 'high_speed', 'Azerbaijan': 'high_speed', 'Singapore': 'technical',
    'United States': 'balanced', 'Mexico': 'balanced', 'Brazil': 'balanced',
    'Las Vegas': 'high_speed', 'Qatar': 'high_speed', 'Abu Dhabi': 'balanced',
}

if 'TrackType' not in historical_races.columns or (historical_races['TrackType'] == 'unknown').any():
    print("⚠️ Fixing TrackType column...")
    historical_races['TrackType'] = historical_races.apply(
        lambda row: CIRCUIT_MAP.get(row['Race'].split()[0], 'balanced') 
        if pd.isna(row.get('TrackType')) or row.get('TrackType') == 'unknown' 
        else row.get('TrackType'),
        axis=1
    )

# Encode circuit types numerically
circuit_encoding = {'high_speed': 0, 'balanced': 1, 'technical': 2}
historical_races['TrackTypeIdx'] = historical_races['TrackType'].map(circuit_encoding)
# Fill any missing values before converting to int
historical_races['TrackTypeIdx'] = historical_races['TrackTypeIdx'].fillna(1).astype(int)  # default to 'balanced'

# Map constructor tier codes
constructor_tier_lookup = dict(zip(constructor_data['Team'], constructor_data['Tier']))
historical_races['TierCode'] = historical_races['Team'].map(constructor_tier_lookup)
# Fill any missing values before converting to int
historical_races['TierCode'] = historical_races['TierCode'].fillna(2).astype(int)  # default to tier 2

# Create driver index mapping
driver_roster = historical_races['Driver'].unique()
driver_index_map = {name: idx for idx, name in enumerate(driver_roster)}
historical_races['DriverIdx'] = historical_races['Driver'].map(driver_index_map)
# Ensure no missing values (shouldn't happen, but safety check)
historical_races['DriverIdx'] = historical_races['DriverIdx'].fillna(0).astype(int)
total_drivers = len(driver_roster)

# Handle DNF cases - assign position 21
historical_races['EffectivePosition'] = historical_races['Position'].copy().astype(float)

# Compute rolling recent performance (last 5 races)
historical_races = historical_races.sort_values(['Driver', 'GlobalRound'])
historical_races['Recent5Avg'] = historical_races.groupby('Driver')['EffectivePosition'].transform(
    lambda series: series.shift(1).rolling(window=5, min_periods=1).mean()
)

# Fill initial values with driver career average
driver_avg_lookup = dict(zip(driver_stats['Driver'], driver_stats['AvgPosition']))
historical_races['Recent5Avg'] = historical_races.apply(
    lambda row: driver_avg_lookup.get(row['Driver'], historical_races['EffectivePosition'].mean()) 
    if pd.isna(row['Recent5Avg']) else row['Recent5Avg'],
    axis=1
)

# Calculate cumulative DNF rate per driver
historical_races['IsDNF'] = (historical_races['Status'] != 'Finished').astype(int)
historical_races['CumDNFRate'] = historical_races.groupby('Driver')['IsDNF'].transform(
    lambda series: series.shift(1).expanding().mean()
)

# Initialize with historical DNF rate
driver_dnf_lookup = dict(zip(driver_stats['Driver'], driver_stats['DNFRate']))
historical_races['CumDNFRate'] = historical_races.apply(
    lambda row: driver_dnf_lookup.get(row['Driver'], 0) 
    if pd.isna(row['CumDNFRate']) else row['CumDNFRate'],
    axis=1
)

In [112]:
# Verify all required columns exist
essential_cols = [
    'Driver', 'Team', 'Race', 'TrackType', 'TrackTypeIdx', 
    'TierCode', 'DriverIdx', 'GridPosition', 'QualifyingPosition',
    'Position', 'EffectivePosition', 'Recent5Avg', 'CumDNFRate', 
    'Status', 'GlobalRound', 'Season'
]

# Save preprocessed dataset
historical_races.to_csv('data/f1_race_data_cleaned.csv', index=False)

In [113]:
print(f"Data preprocessing complete!")
print(f"Total drivers (total_drivers): {total_drivers}")
print(f"Total race entries: {len(historical_races)}")
print(f"\nData preview:")
print(historical_races[['Driver', 'Team', 'Race', 'TierCode', 'TrackTypeIdx', 
                  'EffectivePosition', 'Recent5Avg', 'CumDNFRate']].head(10))

print(f"\nMissing value check:")
print(historical_races[['TierCode', 'TrackTypeIdx', 'Recent5Avg', 'CumDNFRate']].isnull().sum())

print(f"\nConstructor tier distribution:")
print(historical_races.groupby(['Team', 'TierCode']).size().sort_values(ascending=False))


Data preprocessing complete!
Total drivers (total_drivers): 22
Total race entries: 479

Data preview:
      Driver      Team                       Race  TierCode  TrackTypeIdx  \
13   A ALBON  Williams         Pre-Season Testing         1             1   
24   A ALBON  Williams      Australian Grand Prix         1             1   
46   A ALBON  Williams         Chinese Grand Prix         1             1   
68   A ALBON  Williams        Japanese Grand Prix         1             1   
91   A ALBON  Williams         Bahrain Grand Prix         1             0   
108  A ALBON  Williams   Saudi Arabian Grand Prix         1             1   
124  A ALBON  Williams           Miami Grand Prix         1             0   
144  A ALBON  Williams  Emilia Romagna Grand Prix         1             1   
168  A ALBON  Williams          Monaco Grand Prix         1             2   
198  A ALBON  Williams         Spanish Grand Prix         1             1   

     EffectivePosition  Recent5Avg  CumDNFRate  
1

In [114]:
with pm.Model() as hierarchical_f1_model:

    # level 1 priori team
    tier_mean_vals = [-12,-5,0]
    tier_std_val = 2
    team_tier_coef = pm.Normal('beta_team',mu=tier_mean_vals,sigma=tier_std_val,shape=3)
    # level 2 driver ability
    driver_ability_std = pm.HalfNormal('sigma_driver',2.0)
    driver_ability_coef = pm.Normal('gamma_driver',mu =0,sigma = driver_ability_std, shape = total_drivers)
    # level 3 settings
    intercept_param = pm.Normal('alpha', mu=10.6, sigma=2)
    # track_effect_param = pm.Normal('delta_track', mu=0, sigma=3.5)
    starting_grid_coef = pm.Normal('eta_grid', mu=0, sigma=0.5)
    recent_form_coef = pm.Normal('epsilon_trend', mu=0, sigma=3.5)
    reliability_penalty = pm.HalfNormal('zeta_dnf', 3.5)
    # race_noise_param = pm.HalfNormal('sigma_race', 1.0) 

    # Track type effects (3 types: high_speed, balanced, technical)
    track_effect_param = pm.Normal('delta_track', mu=0, sigma=2, shape=3)
    race_noise_param = pm.HalfNormal('sigma_race', 1.0)
    qualifying_coef = pm.Normal('xi_quali', mu=0, sigma=0.8)  

    circuit_encoding = {'high_speed': 0, 'balanced': 1, 'technical': 2}
    historical_races['TrackTypeIdx'] = historical_races['TrackType'].map(circuit_encoding).astype(int)

    # Convert to numpy arrays and ensure integer type
    tier_idx = historical_races['TierCode'].values.astype(int)
    driver_idx = historical_races['DriverIdx'].values.astype(int)
    track_idx = historical_races['TrackTypeIdx'].values.astype(int)
    
    # likelyhood
    predicted_finish = (
        intercept_param +
        team_tier_coef[tier_idx] +
        driver_ability_coef[driver_idx] +  # using driver index!
        starting_grid_coef * historical_races['GridPosition'].values +
        # qualifying_coef * historical_races['QualifyingPosition'].values +
        track_effect_param[track_idx] +  # Track type effect
        recent_form_coef * historical_races['Recent5Avg'].values +
        reliability_penalty * historical_races['CumDNFRate'].values
    )
    observed_finish = pm.Normal(
        'y_obs', 
        mu=predicted_finish, 
        sigma=race_noise_param,
        observed=historical_races['EffectivePosition'].values  # 440 observations!
        # observed=historical_races['QualifyingPosition'].values  # 440 observations!
    )

In [115]:
# ========================================
# Hyperparameter tuning experiments
# Testing different random seeds and acceptance rates
# ==========================================

# # Evaluate convergence across multiple random seeds
# test_seeds = [2020, 2022, 2024, 2026, 2030]
# convergence_results = []

# for seed_val in test_seeds:
#     print(f"\n{'='*60}")
#     print(f"Evaluating seed = {seed_val}")
#     print(f"{'='*60}")
    
#     start_time = time.time()
    
#     with hierarchical_f1_model:
#         trial_trace = pm.sample(
#             draws=4000,
#             tune=2000,
#             chains=2,
#             cores=2,
#             target_accept=0.99,
#             random_seed=seed_val,
#             progressbar=False
#         )
    
#     elapsed_time = time.time() - start_time
    
#     # Convergence diagnostics
#     rhat_stats = az.rhat(trial_trace)
#     ess_stats = az.ess(trial_trace)
    
#     max_rhat_val = max([float(rhat_stats[v].max().values) for v in rhat_stats.data_vars])
#     min_ess_val = min([float(ess_stats[v].min().values) for v in ess_stats.data_vars])
#     divergence_count = int(trial_trace.sample_stats.diverging.sum().values)
    
#     convergence_results.append({
#         'seed': seed_val,
#         'max_rhat': max_rhat_val,
#         'min_ess': min_ess_val,
#         'divergences': divergence_count,
#         'time': elapsed_time
#     })
    
#     print(f"Max R-hat: {max_rhat_val:.4f}")
#     print(f"Min ESS: {min_ess_val:.0f}")
#     print(f"Divergences: {divergence_count}")
#     print(f"Sampling time: {elapsed_time:.1f}s")

# # Summary of seed comparison
# results_df = pd.DataFrame(convergence_results)
# print(f"\n{'='*60}")
# print("SEED COMPARISON SUMMARY")
# print(f"{'='*60}")
# print(results_df)

# print(f"\nESS statistics across seeds:")
# print(f"  Mean ESS: {results_df['min_ess'].mean():.0f}")
# print(f"  Minimum ESS: {results_df['min_ess'].min():.0f}")
# print(f"  Maximum ESS: {results_df['min_ess'].max():.0f}")

# # Select optimal seed
# optimal_seed = results_df.loc[results_df['min_ess'].idxmax(), 'seed']
# print(f"\n✓ Optimal seed identified: {optimal_seed} (ESS = {results_df['min_ess'].max():.0f})")


# # Test different target acceptance rates
# acceptance_rates = [0.95, 0.97, 0.99, 0.998]
# acceptance_results = []

# for target_accept in acceptance_rates:
#     print(f"\n{'='*60}")
#     print(f"Testing target_accept = {target_accept}")
#     print(f"{'='*60}")
    
#     start_time = time.time()
    
#     with hierarchical_f1_model:
#         acceptance_trace = pm.sample(
#             draws=2000,
#             tune=1000,
#             chains=2,
#             cores=2,
#             target_accept=target_accept,
#             random_seed=2024,
#             progressbar=False
#         )
    
#     elapsed_time = time.time() - start_time
    
#     # Diagnostics
#     rhat_check = az.rhat(acceptance_trace)
#     ess_check = az.ess(acceptance_trace)
    
#     max_rhat_result = max([float(rhat_check[v].max().values) for v in rhat_check.data_vars])
#     min_ess_result = min([float(ess_check[v].min().values) for v in ess_check.data_vars])
#     div_count = int(acceptance_trace.sample_stats.diverging.sum().values)
    
#     acceptance_results.append({
#         'target_accept': target_accept,
#         'max_rhat': max_rhat_result,
#         'min_ess': min_ess_result,
#         'divergences': div_count,
#         'time': elapsed_time
#     })
    
#     print(f"Max R-hat: {max_rhat_result:.4f}")
#     print(f"Min ESS: {min_ess_result:.0f}")
#     print(f"Divergences: {div_count}")
#     print(f"Time: {elapsed_time:.1f}s")

# # Summarize acceptance rate comparison
# acceptance_df = pd.DataFrame(acceptance_results)
# print(f"\n{'='*60}")
# print("ACCEPTANCE RATE COMPARISON")
# print(f"{'='*60}")
# print(acceptance_df)

# print(f"\nDivergence statistics:")
# print(f"  Minimum divergences: {acceptance_df['divergences'].min()}")
# print(f"  At acceptance rate: {acceptance_df.loc[acceptance_df['divergences'].idxmin(), 'target_accept']}")


In [117]:
import os
# Final model training with optimal parameters
with hierarchical_f1_model:
    final_trace = pm.sample(
        4000, 
        tune=2000, 
        random_seed=2024, 
        chains=4, 
        cores=4, 
        target_accept=0.998, 
        progressbar=True
    )
    trace_file_path = './model/f1_trace.nc'
    if os.path.exists(trace_file_path):
        os.remove(trace_file_path)
        print(f"✓ Removed existing trace file")
    
    final_trace.to_netcdf(trace_file_path)
    print(f"✓ Trace successfully saved to '{trace_file_path}'")


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_team, sigma_driver, gamma_driver, alpha, eta_grid, epsilon_trend, zeta_dnf, delta_track, sigma_race, xi_quali]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 70 seconds.


✓ Removed existing trace file
✓ Trace successfully saved to './model/f1_trace.nc'


In [118]:
# Load and diagnose final model trace
final_trace = az.from_netcdf('./model/f1_trace.nc')

print("="*60)
print("MODEL CONVERGENCE DIAGNOSTICS")
print("="*60)

# R-hat convergence metric
rhat_diagnostic = az.rhat(final_trace)
rhat_values = [float(rhat_diagnostic[var].max().values) for var in rhat_diagnostic.data_vars]
max_rhat_final = max(rhat_values)

print(f"\nMaximum R-hat: {max_rhat_final:.4f}")

if max_rhat_final > 1.01:
    print("⚠️ WARNING: Chains have NOT converged!")
else:
    print("✓ Convergence achieved")

# Effective Sample Size
ess_diagnostic = az.ess(final_trace)
ess_values = [float(ess_diagnostic[var].min().values) for var in ess_diagnostic.data_vars]
min_ess_final = min(ess_values)

print(f"Minimum ESS: {min_ess_final:.0f}")

if min_ess_final < 400:
    print("⚠️ WARNING: Insufficient effective samples")
else:
    print("✓ Adequate effective sample size")

# Divergence check
divergence_total = int(final_trace.sample_stats.diverging.sum().values)
print(f"Total divergences: {divergence_total}")

if divergence_total > 0:
    print(f"⚠️ {divergence_total} divergent transitions detected")
else:
    print("✓ No divergences detected")

# Overall assessment
print("\n" + "="*60)
if max_rhat_final < 1.01 and min_ess_final > 400:
    print("✓ MODEL IS RELIABLE")
    print("Proceed with posterior analysis!")
elif max_rhat_final > 1.01:
    print("❌ REQUIRES RESAMPLING with increased target_accept")
elif min_ess_final < 100:
    print("❌ INSUFFICIENT SAMPLES - increase draws")
else:
    print("⚠️ MODEL ACCEPTABLE but improvable")


MODEL CONVERGENCE DIAGNOSTICS

Maximum R-hat: 1.0047
✓ Convergence achieved
Minimum ESS: 455
✓ Adequate effective sample size
Total divergences: 0
✓ No divergences detected

✓ MODEL IS RELIABLE
Proceed with posterior analysis!


In [119]:
# Posterior predictive validation
historical_races = pd.read_csv('./data/f1_race_data_cleaned.csv')

with hierarchical_f1_model:
    posterior_predictions = pm.sample_posterior_predictive(final_trace, random_seed=42)

predicted_positions = posterior_predictions.posterior_predictive['y_obs'].values
actual_positions = historical_races['EffectivePosition'].values

predicted_mean = predicted_positions.mean(axis=(0, 1))

model_r2 = r2_score(actual_positions, predicted_mean)
model_mae = mean_absolute_error(actual_positions, predicted_mean)

print(f"\nModel performance:")
print(f"R² = {model_r2:.3f}")
print(f"MAE = {model_mae:.2f} positions")

Sampling: [y_obs]


Output()


Model performance:
R² = 0.490
MAE = 3.12 positions
